# Unidad 3 — Variables aleatorias

Una **variable aleatoria** es una función $X: \Omega \to \mathbb{R}$ que le pone un
número a cada resultado del experimento. Eso nos deja:

- Hablar de **distribuciones** (la "forma" de probabilidades sobre los valores).
- Definir **esperanza** y **varianza** como números.
- Pasar de modelos discretos a continuos sin cambiar de lenguaje.

Las distribuciones simbólicas viven en `src/symbolic/moments.py`; las numéricas en
`src/distributions/`.


## Importaciones

In [ ]:
from core import (
    BinomialParams,
    NormalParams,
    PoissonParams,
    Settings,
)
from distributions import (
    DensityGridInput,
    MomentsInput,
    ProbabilityMassInput,
    QuantileInput,
    TailProbabilityInput,
    compute_numeric_moments,
    evaluate_density_grid,
    evaluate_probability_mass,
    make_binomial,
    make_normal,
    make_poisson,
    quantile_of_continuous,
    tail_probability_of_continuous,
)
from exercises import NumericAnswerInput, verify_numeric_answer
from symbolic import (
    compute_binomial_moments,
    compute_poisson_moments,
    standardize_normal,
)
from visualization import (
    DensityChartInput,
    ProbabilityMassChartInput,
    chart_density,
    chart_probability_mass,
)
from widgets import (
    ContinuousDistributionExplorerInput,
    DiscreteDistributionExplorerInput,
    build_continuous_distribution_explorer,
    build_discrete_distribution_explorer,
)

In [ ]:
settings = Settings()

## Concreto (discreto): Binomial(10, 0.3)

Tiramos una moneda sesgada 10 veces. $X$ cuenta cuántas caras salen.
$X \sim \text{Bin}(n=10, p=0{,}3)$.

**Intuición Singapur.** $E[X] = np = 3$ es el "centro de gravedad" si dibujamos las
barras del PMF. Movés $p$ y el centro se mueve linealmente.


In [ ]:
binomial_distribution = make_binomial(BinomialParams(trials=10, success_probability=0.3))
probability_mass = evaluate_probability_mass(ProbabilityMassInput(distribution=binomial_distribution))
chart_probability_mass(ProbabilityMassChartInput(probability_mass=probability_mass, settings=settings))

### Momentos simbólicos

Los mismos momentos pero como expresiones simbólicas — útiles para *demostrar* las
fórmulas "$np$" y "$np(1-p)$" en clase sin escribirlas dos veces.


In [ ]:
binomial_moments_symbolic = compute_binomial_moments(BinomialParams(trials=10, success_probability=0.3))

binomial_moments_numeric = compute_numeric_moments(MomentsInput(distribution=binomial_distribution))
binomial_moments_numeric

## Concreto (continuo): Normal(0, 1)

Pasamos del PMF (barras) al PDF (curva). La **probabilidad ya no vive en los puntos**:
vive en el **área debajo de la curva**.

$$ P(a \le X \le b) = \int_a^b f_X(x)\,dx $$


In [ ]:
standard_normal_distribution = make_normal(NormalParams(mean=0.0, standard_deviation=1.0))
density = evaluate_density_grid(DensityGridInput(distribution=standard_normal_distribution, settings=settings))
chart_density(DensityChartInput(density_grid=density, settings=settings))

### Estandarización

Cualquier Normal $X \sim \mathcal{N}(\mu, \sigma^2)$ se convierte en
$Z \sim \mathcal{N}(0, 1)$ con un cambio de variable lineal:


In [ ]:
standardize_normal(NormalParams(mean=170.0, standard_deviation=8.0)).formula

Es la **misma transformación** que usamos en Unidad 1 con el $z$-score. La diferencia
es que ahora sabemos qué distribución tiene el resultado: $\mathcal{N}(0, 1)$.


## Pictórico: probabilidades como áreas

Pregunta concreta: si las alturas en una población se distribuyen $\mathcal{N}(170, 8^2)$,
¿cuál es la probabilidad de medir entre 165 y 180 cm?


In [ ]:
heights_distribution = make_normal(NormalParams(mean=170.0, standard_deviation=8.0))
probability_between = tail_probability_of_continuous(
    TailProbabilityInput(distribution=heights_distribution, lower_bound=165.0, upper_bound=180.0)
)
probability_between

## Cuantiles: invertir la pregunta

En lugar de fijar $x$ y leer probabilidad, fijamos probabilidad y leemos $x$. El
**percentil 90** es "el valor por debajo del cual queda el 90% de la población".


In [ ]:
percentile_ninety = quantile_of_continuous(QuantileInput(distribution=heights_distribution, probability=0.90))
percentile_ninety

## Exploración interactiva — distribuciones continuas

Cambiamos familia, parámetros y un intervalo $[x_{\min}, x_{\max}]$. La probabilidad
se actualiza en vivo.


In [ ]:
build_continuous_distribution_explorer(ContinuousDistributionExplorerInput(settings=settings))

## Exploración interactiva — distribuciones discretas

Pasamos de Binomial a Poisson moviendo el dropdown. **Intuición:** cuando $n$ es grande y
$p$ es chico (con $np = \lambda$ moderado), Binomial $\approx$ Poisson.


In [ ]:
build_discrete_distribution_explorer(DiscreteDistributionExplorerInput(settings=settings))

## Poisson para eventos raros

Un call center recibe en promedio $\lambda = 4$ llamadas por minuto. ¿Probabilidad de
recibir exactamente 6 en un minuto?

$$ P(K = 6) = \frac{\lambda^6 e^{-\lambda}}{6!} $$


In [ ]:
poisson_distribution = make_poisson(PoissonParams(rate=4.0))
poisson_mass = evaluate_probability_mass(
    ProbabilityMassInput(distribution=poisson_distribution, lower_outcome=0, upper_outcome=12)
)
chart_probability_mass(ProbabilityMassChartInput(probability_mass=poisson_mass, settings=settings))

## Ejercicio 1 — Estandarización

$X \sim \mathcal{N}(170, 8^2)$. Calcular $P(X \le 178)$ usando $Z = (X - 170)/8$.

$$ P(X \le 178) = P(Z \le 1) \approx 0{,}8413 $$


In [ ]:
expected_probability = tail_probability_of_continuous(
    TailProbabilityInput(distribution=heights_distribution, upper_bound=178.0)
).probability

student_answer = 0.8413
verify_numeric_answer(
    NumericAnswerInput(
        student_answer=student_answer,
        expected_answer=expected_probability,
        absolute_tolerance=1e-3,
    )
)

## Ejercicio 2 — Esperanza de una Poisson

Si $K \sim \text{Poisson}(\lambda = 3{,}5)$, ¿cuál es $E[K]$?

**Idea.** $E[K] = \lambda = 3{,}5$. La Poisson **es** su tasa.


In [ ]:
expected_expectation = compute_poisson_moments(PoissonParams(rate=3.5)).expectation

student_answer = 3.5
verify_numeric_answer(NumericAnswerInput(student_answer=student_answer, expected_answer=float(expected_expectation)))

## Para llevarse

- **PMF (discreto) ↔ PDF (continuo)**: la probabilidad pasa de "barras" a "áreas".
- Estandarizar = trasladar y reescalar para hablar todos el mismo idioma.
- La Normal estándar **no** describe a la población; es la unidad de medida con la que la describimos.
- Bin($n$ grande, $p$ chico) $\to$ Poisson($\lambda = np$): el discreto colapsa al "raro".
